# Situation #

We have tons of DES SNIa that are needed to be matched up to hosts and we need to upload them to the BLAST server and stuff. So we manuall removed the iaua name containing SNIa (now that we can resort to processing things in BLAST with just using CID's, lets see what happens). 


No need to do anything with TNS server at all. Just living life and stuff. 

### Conceptual reasoning

- **zHEL (heliocentric redshift):**  
  This is the directly measured spectroscopic redshift of the supernova spectrum in the heliocentric frame (corrected only for the Earth’s orbital motion around the Sun). It’s the *raw* transient redshift.

- **zCMB:**  
  That’s the same redshift, but transformed into the frame of the cosmic microwave background dipole (i.e., correcting for our solar system’s motion relative to the CMB). This is useful for **cosmological analysis** (Hubble diagrams), not for identifying or cross-matching a transient.

- **zHD:**  
  That’s a further **Hubble-diagram adjusted redshift**, where peculiar velocity models and flow corrections are folded in. It’s even more “analysis-specific,” not a directly observed value.

---

For BLAST (or any upload that’s just meant to identify the transient itself with its own spectrum and coordinates), you want the **measured, heliocentric redshift**:  
👉 **zHEL is the observational property of the transient.**

Everything else (`zCMB`, `zHD`, host redshifts) are **cosmology tools**, not the transient’s intrinsic spectroscopic measurement.

In [1]:
import pandas as pd
import pandas as pd
from astropy.table import Table

# point this to your actual file
in_path  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua.csv"

df = pd.read_csv(in_path)

for col in df.columns:
    print(col)


CID
CIDint
IDSURVEY
TYPE
zHEL
zHELERR
zCMB
zCMBERR
zHD
zHDERR
VPEC
VPECERR
MWEBV
HOST_ZSPEC
HOST_ZSPECERR
HOST_RA
HOST_DEC
HOST_ANGSEP
HOST_DDLR
HOST_LOGMASS
HOST_LOGMASS_ERR
HOST_COLOR
HOST_COLOR_ERR
PKMJD
PKMJDERR
x1
x1ERR
c
cERR
mB
mBERR
mB_corr
x0
x0ERR
COV_x1_c
COV_x1_x0
COV_c_x0
NDOF
FITPROB
PROB_SCONE
PROB_SNIRFV19
PROB_SNNDESCC
PROB_SNNJ17
PROB_SNNV19
MU
MUERR_FINAL
PROBCC_BEAMS
biasCor_mu
biasCor_muCOVSCALE
biasCor_muCOVADD


So it turns out that there are no RA and DEC measurements for the SNIa that were part of the DES data release and we must look at the DES .FITS files in order to open stuff up. 

In [4]:
# ----------------------------
# Paths
# ----------------------------
DES_HEAD_PATH   = "/Users/pittsburghgraduatestudent/repos/data/BLAST_API_query_data/DES-SN5YR-1.2/0_DATA/DES-SN5YR_DES/DES-SN5YR_DES_HEAD.FITS.gz"
LOWZ_HEAD_PATH  = "/Users/pittsburghgraduatestudent/repos/data/BLAST_API_query_data/DES-SN5YR-1.2/0_DATA/DES-SN5YR_LOWZ/DES-SN5YR_LOWZ_HEAD.FITS.gz"
FND_HEAD_PATH   = "/Users/pittsburghgraduatestudent/repos/data/BLAST_API_query_data/DES-SN5YR-1.2/0_DATA/DES-SN5YR_Foundation/DES-SN5YR_Foundation_HEAD.FITS.gz"

CSV_PATH        = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua.csv"
OUT_PATH        = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/SNIa_no_hostC_yes_SNIaC_DES_LOWZ_FOUNDATION.csv"

# ----------------------------
# Load metadata CSV (with CIDs) and normalize key
# ----------------------------
meta_df = pd.read_csv(CSV_PATH)
meta_df = meta_df.rename(columns={"CID": "SNID"}).copy()
# Normalize SNID to string for robust joins
meta_df["SNID"] = meta_df["SNID"].astype(str).str.strip()

# ----------------------------
# Helper to read a HEAD table and return SNID, RA, DEC as strings->float
# ----------------------------
def load_coords(head_path, ra_col="RA", dec_col="DEC"):
    t = Table.read(head_path, hdu=1)
    df = t.to_pandas()

    # Normalize SNID type/bytes → str
    if "SNID" in df.columns:
        if df["SNID"].dtype == object and len(df) > 0 and isinstance(df["SNID"].iloc[0], (bytes, bytearray)):
            df["SNID"] = df["SNID"].str.decode("utf-8", errors="ignore")
        df["SNID"] = df["SNID"].astype(str).str.strip()
    else:
        raise KeyError(f"'SNID' not found in {head_path}")

    # Keep only what we need and rename to avoid collisions
    out = df[["SNID", ra_col, dec_col]].copy()
    out.columns = ["SNID", "RA_SRC", "DEC_SRC"]
    return out

# ----------------------------
# Load coordinate sources
# ----------------------------
des_coords  = load_coords(DES_HEAD_PATH)
lowz_coords = load_coords(LOWZ_HEAD_PATH)
fnd_coords  = load_coords(FND_HEAD_PATH)

# Tag source to help debug (optional)
des_coords["SRC"]  = "DES"
lowz_coords["SRC"] = "LOWZ"
fnd_coords["SRC"]  = "FOUNDATION"

# ----------------------------
# Merge: left-join meta with DES, then fill from LOWZ, then FOUNDATION
# Priority: DES → LOWZ → FOUNDATION
# ----------------------------
merged = meta_df.merge(des_coords, on="SNID", how="left")
merged = merged.merge(lowz_coords, on="SNID", how="left", suffixes=("", "_LOWZ"))
merged = merged.merge(fnd_coords, on="SNID", how="left", suffixes=("", "_FND"))

# Build final RA/DEC with cascade
final_ra  = merged["RA_SRC"].combine_first(merged["RA_SRC_LOWZ"]).combine_first(merged["RA_SRC_FND"])
final_dec = merged["DEC_SRC"].combine_first(merged["DEC_SRC_LOWZ"]).combine_first(merged["DEC_SRC_FND"])

# Attach to output frame
out_cols = meta_df.columns.tolist()  # keep all original meta columns
out = merged[out_cols].copy()
out["RA"]  = final_ra
out["DEC"] = final_dec

# Optional: track where each coordinate came from
src = merged["SRC"].combine_first(merged["SRC_LOWZ"]).combine_first(merged["SRC_FND"])
out["COORD_SOURCE"] = src

# ----------------------------
# Save and quick report
# ----------------------------
out.to_csv(OUT_PATH, index=False)

n_total = len(out)
n_have  = out["RA"].notna().sum()
print(f"Saved → {OUT_PATH}")
print(f"Matched RA/DEC for {n_have}/{n_total} rows "
      f"({100.0 * n_have/n_total:.1f}%).")
out.head()

Saved → /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/SNIa_no_hostC_yes_SNIaC_DES_LOWZ_FOUNDATION.csv
Matched RA/DEC for 1635/1635 rows (100.0%).


,SNID,CIDint,IDSURVEY,TYPE,zHEL,zHELERR,zCMB,zCMBERR,zHD,zHDERR,...,PROB_SNNV19,MU,MUERR_FINAL,PROBCC_BEAMS,biasCor_mu,biasCor_muCOVSCALE,biasCor_muCOVADD,RA,DEC,COORD_SOURCE
0,1246275,1246275,10,0,0.24651,0.001,0.24605,0.001,0.24605,0.00160,...,1.0000,40.5938,0.0968,0.0,0.0341,1.0,0.0050,54.647026,-26.401205,DES
1,1246281,1246281,10,0,0.33600,0.001,0.33549,0.001,0.33549,0.00167,...,1.0000,41.2263,0.1360,0.0,-0.0492,1.0,0.0136,53.725414,-27.622061,DES
2,1246314,1246314,10,0,0.38388,0.001,0.38337,0.001,0.38337,0.00171,...,0.9998,41.6383,0.2332,0.0,0.0502,1.0,0.0348,54.836567,-26.640186,DES
3,1246527,1246527,10,0,0.32184,0.001,0.32078,0.001,0.32078,0.00166,...,1.0000,41.1991,0.1503,0.0,-0.0511,1.0,0.0173,36.018959,-5.016278,DES
4,1246529,1246529,10,0,0.49797,0.001,0.49677,0.001,0.49677,0.00180,...,1.0000,42.1471,0.1618,0.0,-0.0485,1.0,0.0095,36.113865,-4.944012,DES


In [8]:
# point this to your actual file
in_path  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/SNIa_no_hostC_yes_SNIaC_DES_LOWZ_FOUNDATION.csv"
out_path = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua_cleaned_for_BLAST.csv"

df = pd.read_csv(in_path)

# keep only the requested columns
subset = df[["SNID", "RA", "DEC", "zHEL", "TYPE"]]

# overwrite TYPE column with "None"
subset["TYPE"] = "None"

# save
subset.to_csv(out_path, index=False)

print("Saved:", out_path)
print(subset.head())

Saved: /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua_cleaned_for_BLAST.csv
      SNID         RA        DEC     zHEL  TYPE
0  1246275  54.647026 -26.401205  0.24651  None
1  1246281  53.725414 -27.622061  0.33600  None
2  1246314  54.836567 -26.640186  0.38388  None
3  1246527  36.018959  -5.016278  0.32184  None
4  1246529  36.113865  -4.944012  0.49797  None


/var/folders/71/hv72gkrs7g59ty6664549kjr0000gr/T/ipykernel_21184/1497100397.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subset["TYPE"] = "None"


In [10]:
import pandas as pd
import os

# Input is the single cleaned file you just created
in_path  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/DES_no_iaua_cleaned_for_BLAST.csv"
out_dir  = "/Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks"

# Make sure output directory exists
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(in_path)

# Split into chunks of 200 rows each
chunk_size = 200
n_chunks = (len(df) + chunk_size - 1) // chunk_size

for i in range(n_chunks):
    start = i * chunk_size
    end   = start + chunk_size
    chunk = df.iloc[start:end]
    out_path = os.path.join(out_dir, f"DES_no_iaua_cleaned_for_BLAST_part{i+1}.csv")
    chunk["TYPE"] = "None"  # Ensure TYPE column is set to "None"
    chunk.to_csv(out_path, index=False)
    print(f"Saved {out_path} with {len(chunk)} rows")

print(f"Done! Total chunks: {n_chunks}")

Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part1.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part2.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part3.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part4.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part5.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part6.csv with 200 rows
Saved /Users/pittsburghgraduatestudent/repos/first_paper_blast_webapi/data/blast_chunks/DES_no_iaua_cleaned_for_BLAST_part7.csv with 200 rows
Saved 

/var/folders/71/hv72gkrs7g59ty6664549kjr0000gr/T/ipykernel_21184/126872605.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunk["TYPE"] = "None"  # Ensure TYPE column is set to "None"
